# Notebook 4 — Deep learning reconstruction

*Part of **MRI Reconstruction from k-space to Deep Learning: A Hands-on Introduction***

Notebook 3 ended with the limits of compressed sensing: λ must be tuned, it is slow, and its idea of "what a real image looks like" (sparse Haar wavelets) is a hand-made rule. Deep learning replaces that rule with one **learned from examples**. In this notebook you will train two networks and compare them with everything you have built so far.

**Learning objectives.** By the end of this notebook you will be able to:

1. Explain **supervised learning** for reconstruction: training on pairs of undersampled and fully sampled images.
2. Build a simulated training set, and explain why a network needs many different examples.
3. Train a **U-Net** that removes undersampling artifacts from zero-filled images.
4. Build an **unrolled network** that alternates learned steps with **data consistency**, and explain why that helps.
5. Compare zero-filling, compressed sensing and both networks on accuracy and speed.

**Prerequisites:** Notebooks 1–3, and a basic idea of a neural network (layers with trainable weights, a loss, training by gradient descent). No PyTorch experience is needed; every step is explained.

**Time:** about 90–120 minutes, including about 10 minutes of training on a CPU. **Tip:** in Colab, choose *Runtime > Change runtime type > T4 GPU* first; training then takes only a minute or two.

As always, answer every **🤔 Predict** question before running the cell.

## 1. Setup

We now also import **PyTorch** (`torch`), the deep learning library, which is preinstalled in Colab. The second cell repeats the helpers from earlier notebooks, now able to work on a whole stack of images at once, plus the compressed sensing code from Notebook 3 so we can compare with it.

In [ ]:
import time
import urllib.request
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from skimage.data import shepp_logan_phantom
from skimage.transform import resize

plt.rcParams["figure.dpi"] = 100
COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # line colors, always used in this order

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(0)
print(f"Using {device}." + ("" if device.type == "cuda" else
      "  Training will take about 10 minutes in total. For a faster run: Runtime > Change runtime type > T4 GPU."))

In [ ]:
# ---- From Notebooks 1-3 (now working on stacks of images too) ------------------
def fft2c(image):
    """Centered, orthonormal 2D FFT over the last two axes: image(s) -> k-space."""
    return np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(image, axes=(-2, -1)), norm="ortho"), axes=(-2, -1))


def ifft2c(kspace):
    """Centered, orthonormal inverse 2D FFT over the last two axes: k-space -> image(s)."""
    return np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(kspace, axes=(-2, -1)), norm="ortho"), axes=(-2, -1))


def show(images, titles, cmap="gray", vmin=None, vmax=None):
    """Show images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=(3.4 * n, 3.6))
    for ax, im, title in zip(np.atleast_1d(axes), images, titles):
        ax.set_title(title)
        ax.imshow(im, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


def nrmse(recon, reference):
    """Normalized root-mean-square error: 0 is perfect, 1 is as large as the image itself."""
    return np.linalg.norm(recon - reference) / np.linalg.norm(reference)


def tidy(ax):
    """Quiet grid and axes for line plots."""
    ax.grid(alpha=0.25, linewidth=0.6)
    ax.spines[["top", "right"]].set_visible(False)


def variable_density_lines(N, R, center_fraction=0.08, decay=3, seed=0):
    """N // R lines: a fully sampled center, the rest random and more likely near the center."""
    rng = np.random.default_rng(seed)
    n_total = N // R
    n_center = int(round(N * center_fraction))
    lines = np.zeros(N, dtype=bool)
    lines[N // 2 - n_center // 2 : N // 2 - n_center // 2 + n_center] = True
    candidates = np.flatnonzero(~lines)
    distance = np.abs(candidates - N // 2) / (N / 2)
    probability = (1 - distance) ** decay + 1e-6
    lines[rng.choice(candidates, size=n_total - n_center, replace=False,
                     p=probability / probability.sum())] = True
    return lines


# ---- Compressed sensing from Notebook 3, for comparison --------------------------
SQRT2 = np.sqrt(2)


def haar2(x, levels=4):
    c = x.astype(complex).copy()
    n = c.shape[0]
    for _ in range(levels):
        a = c[:n, :n]
        a = np.hstack([(a[:, 0::2] + a[:, 1::2]) / SQRT2, (a[:, 0::2] - a[:, 1::2]) / SQRT2])
        a = np.vstack([(a[0::2, :] + a[1::2, :]) / SQRT2, (a[0::2, :] - a[1::2, :]) / SQRT2])
        c[:n, :n] = a
        n //= 2
    return c


def ihaar2(c, levels=4):
    x = c.copy()
    n = x.shape[0] // 2 ** (levels - 1)
    for _ in range(levels):
        a = x[:n, :n]
        h = n // 2
        b = np.empty_like(a)
        b[0::2, :] = (a[:h, :] + a[h:, :]) / SQRT2
        b[1::2, :] = (a[:h, :] - a[h:, :]) / SQRT2
        a2 = np.empty_like(a)
        a2[:, 0::2] = (b[:, :h] + b[:, h:]) / SQRT2
        a2[:, 1::2] = (b[:, :h] - b[:, h:]) / SQRT2
        x[:n, :n] = a2
        n *= 2
    return x


def soft_threshold(c, lam):
    magnitude = np.abs(c)
    return c * np.maximum(magnitude - lam, 0) / np.maximum(magnitude, 1e-12)


def cs_reconstruct(kspace_measured, lines, lam=0.01, n_iter=200, levels=4):
    n = kspace_measured.shape[0]
    coarse = np.zeros((n, n), dtype=bool)
    coarse[: n // 2**levels, : n // 2**levels] = True
    x = ifft2c(kspace_measured)
    for _ in range(n_iter):
        k = np.where(lines[:, None], kspace_measured, fft2c(x))
        x = ifft2c(k)
        w = haar2(x, levels)
        x = ihaar2(np.where(coarse, w, soft_threshold(w, lam)), levels)
    return x

## 2. Learning from examples

**Supervised learning** for reconstruction works like this:

1. Collect many **fully sampled** scans: the "ground truth".
2. Undersample each one *in software*, exactly as the fast scan would, and compute its zero-filled image.
3. Train a network to turn each zero-filled image back into its ground truth.
4. On a new, really undersampled scan, the trained network does the reconstruction.

The network must learn what real images look like, so it needs **many different** examples. A single image would simply be memorized.

Real projects use thousands of real fully sampled scans, such as the public fastMRI dataset. To keep this notebook fast and download-free, we generate random **head-like phantoms**: a skull ring, a brain, random internal structures of different sizes and brightness, and a few small "lesions". Every phantom is different.

In [ ]:
def random_phantom(rng, N=128, oversample=2):
    """A random head-like phantom: a bright skull ring, a darker brain,
    4-8 internal structures and 1-3 small 'lesions'. Values between 0 and 1."""
    M = N * oversample                                  # draw at higher resolution, then average down
    y, x = np.mgrid[-1:1:M * 1j, -1:1:M * 1j]
    img = np.zeros((M, M))

    def ellipse(cx, cy, a, b, angle, value):
        c, s = np.cos(angle), np.sin(angle)
        xr = (x - cx) * c + (y - cy) * s
        yr = -(x - cx) * s + (y - cy) * c
        img[(xr / a) ** 2 + (yr / b) ** 2 <= 1] += value

    def random_point_inside(spread):
        r, phi = rng.uniform(0, spread), rng.uniform(0, 2 * np.pi)
        return cx + r * (a - t) * 0.8 * np.cos(phi), cy + r * (b - t) * 0.8 * np.sin(phi)

    # head outline: skull ring around a darker brain
    cx, cy = rng.uniform(-0.05, 0.05, 2)
    a, b = rng.uniform(0.6, 0.75), rng.uniform(0.8, 0.92)
    angle = rng.uniform(-0.2, 0.2)
    t = rng.uniform(0.04, 0.07)                         # skull thickness
    ellipse(cx, cy, a, b, angle, 1.0)
    ellipse(cx, cy, a - t, b - t, angle, rng.uniform(0.15, 0.3) - 1.0)

    # internal structures of different sizes and brightness
    for _ in range(rng.integers(4, 9)):
        ex, ey = random_point_inside(0.6)
        size = rng.uniform(0.04, 0.25)
        ellipse(ex, ey, size * rng.uniform(0.4, 1.0), size, rng.uniform(0, np.pi), rng.uniform(-0.2, 0.25))

    # a few small bright or dark "lesions"
    for _ in range(rng.integers(1, 4)):
        ex, ey = random_point_inside(0.7)
        size = rng.uniform(0.015, 0.04)
        ellipse(ex, ey, size, size * rng.uniform(0.6, 1.0), rng.uniform(0, np.pi),
                rng.uniform(0.1, 0.3) * rng.choice([-1, 1]))

    img = np.clip(img, 0, 1)
    return img.reshape(N, oversample, N, oversample).mean(axis=(1, 3))

In [ ]:
examples = np.stack([random_phantom(np.random.default_rng(100 + i)) for i in range(4)])
show(list(examples), [f"random phantom {i + 1}" for i in range(4)], vmin=0, vmax=1)

Now generate the full dataset: **1000 training**, **100 validation** and **100 test** phantoms. Each one is undersampled with the same fixed variable-density R = 4 mask, just as a scanner would use one fixed protocol.

We use 128 × 128 images (half the size of Notebooks 1–3) so that training is fast. We also keep the **Shepp-Logan phantom** aside as an extra test image: the networks never see it during training, and it looks somewhat different from our random phantoms.

In [ ]:
N, R = 128, 4
N_TRAIN, N_VAL, N_TEST = 1000, 100, 100

rng = np.random.default_rng(0)
start = time.time()
truth = np.stack([random_phantom(rng, N) for _ in range(N_TRAIN + N_VAL + N_TEST)])
print(f"Generated {len(truth)} phantoms in {time.time() - start:.0f} s")

lines = variable_density_lines(N, R)                    # one fixed sampling pattern, as on a scanner
measured = fft2c(truth) * lines[:, None]                # undersampled k-space (unmeasured lines = 0)
zero_filled = ifft2c(measured)                          # network input: complex zero-filled images

train, val, test = slice(0, N_TRAIN), slice(N_TRAIN, N_TRAIN + N_VAL), slice(N_TRAIN + N_VAL, None)

# An extra test image the networks never see in any form: the Shepp-Logan phantom
sl_truth = resize(shepp_logan_phantom(), (N, N), anti_aliasing=True)
sl_measured = fft2c(sl_truth) * lines[:, None]
sl_zero_filled = ifft2c(sl_measured)
print(f"{lines.sum()} of {N} lines measured (R = {R})")

- The **training** images are used to adjust the network's weights.
- The **validation** images are checked after every epoch, to see whether the network also improves on images it does not train on.
- The **test** images are used only once, at the end, for the final comparison. Never tune anything on them.

Networks work with real numbers, so each complex image becomes **two channels**: its real and imaginary parts.

In [ ]:
def to_tensor(c):
    """Complex images (B, N, N) -> real tensor (B, 2, N, N): channel 0 = real part, channel 1 = imaginary part."""
    return torch.tensor(np.stack([c.real, c.imag], axis=1), dtype=torch.float32)


def to_complex(t):
    """Inverse of to_tensor: (B, 2, N, N) -> complex NumPy array (B, N, N)."""
    t = t.detach().cpu().numpy()
    return t[:, 0] + 1j * t[:, 1]


X = to_tensor(zero_filled)                               # inputs
Y = to_tensor(truth.astype(complex))                     # targets: the true images (imaginary part 0)
K = torch.tensor(measured, dtype=torch.complex64)        # measured k-space, needed for data consistency
MASK = torch.tensor(lines[:, None], device=device)       # True on measured lines

print("inputs:", tuple(X.shape), " targets:", tuple(Y.shape))

## 3. The U-Net

The **U-Net** (Ronneberger et al., 2015) is the most widely used network in medical imaging. It has two halves:

- The **encoder** repeatedly applies convolutions and halves the image size (128 → 64 → 32 → 16 pixels). Each step sees a larger area, so it can recognize larger patterns, such as long aliasing streaks.
- The **decoder** doubles the size back again. At each size, a **skip connection** passes in the encoder's features of the same size, so fine detail is not lost.

Our U-Net is **residual**: it predicts a *correction* that is added to the zero-filled input. Learning "what is wrong with this image" is easier than drawing the whole image from scratch.

In [ ]:
def conv_block(c_in, c_out):
    """Two 3x3 convolutions, each followed by ReLU."""
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.ReLU(inplace=True),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.ReLU(inplace=True),
    )


class UNet(nn.Module):
    """A small U-Net that predicts a correction to the zero-filled image."""

    def __init__(self, channels=2, base=16):
        super().__init__()
        # encoder: features at 128, 64 and 32 pixels
        self.enc1 = conv_block(channels, base)
        self.enc2 = conv_block(base, 2 * base)
        self.enc3 = conv_block(2 * base, 4 * base)
        # bottleneck at 16 pixels
        self.bottom = conv_block(4 * base, 8 * base)
        # decoder: upsample, join with the matching encoder features (skip connection), convolve
        self.up3, self.dec3 = nn.ConvTranspose2d(8 * base, 4 * base, 2, stride=2), conv_block(8 * base, 4 * base)
        self.up2, self.dec2 = nn.ConvTranspose2d(4 * base, 2 * base, 2, stride=2), conv_block(4 * base, 2 * base)
        self.up1, self.dec1 = nn.ConvTranspose2d(2 * base, base, 2, stride=2), conv_block(2 * base, base)
        self.out = nn.Conv2d(base, channels, 1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(F.max_pool2d(e1, 2))
        e3 = self.enc3(F.max_pool2d(e2, 2))
        b = self.bottom(F.max_pool2d(e3, 2))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return x + self.out(d1)          # residual: input + learned correction


def count_parameters(model):
    return sum(p.numel() for p in model.parameters())

In [ ]:
unet = UNet().to(device)
print(f"U-Net with {count_parameters(unet):,} trainable weights")

## 4. Training

Training repeats the same small loop many times:

1. Take a **batch** of 16 training images.
2. Run them through the network.
3. Measure the **loss**: the L1 loss, the average absolute difference from the ground truth (common in MRI reconstruction because it keeps edges sharper than the squared error).
4. **Backpropagation** computes how each weight should change to lower the loss, and the **Adam** optimizer makes a small change.

One pass through all 1000 training images is an **epoch**. The functions below do this, report the loss on the training and validation images after each epoch, and reconstruct new images with a trained model.

If you are short on time, set `USE_PRETRAINED = True` to download weights that were trained with exactly these settings instead of training yourself.

In [ ]:
def run_model(model, x, k):
    """Run a model on a batch. Unrolled networks also need the measured k-space."""
    return model(x, k) if getattr(model, "uses_kspace", False) else model(x)


def train_model(model, epochs, batch_size=16, lr=1e-3, n_train=N_TRAIN, seed=0):
    """Train with the L1 loss and the Adam optimizer. Returns the loss history."""
    torch.manual_seed(seed)
    model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    order = np.arange(n_train)
    history = {"train": [], "val": []}
    for epoch in range(epochs):
        start = time.time()
        model.train()
        np.random.default_rng(seed + epoch).shuffle(order)
        total = 0.0
        for i in range(0, n_train, batch_size):
            b = order[i : i + batch_size]
            x, y, k = X[b].to(device), Y[b].to(device), K[b].to(device)
            loss = F.l1_loss(run_model(model, x, k), y)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total += loss.item() * len(b)
        model.eval()
        with torch.no_grad():
            val_loss = F.l1_loss(run_model(model, X[val].to(device), K[val].to(device)), Y[val].to(device)).item()
        history["train"].append(total / n_train)
        history["val"].append(val_loss)
        print(f"epoch {epoch + 1}/{epochs}: train loss {total / n_train:.4f}, "
              f"validation loss {val_loss:.4f}  ({time.time() - start:.0f} s)")
    return history


def predict(model, x_complex, k_measured):
    """Reconstruct magnitude images (B, N, N) from zero-filled complex images and measured k-space."""
    model.eval()
    with torch.no_grad():
        x = to_tensor(x_complex).to(device)
        k = torch.tensor(k_measured, dtype=torch.complex64).to(device)
        return np.abs(to_complex(run_model(model, x, k)))


def plot_history(history, title):
    fig, ax = plt.subplots(figsize=(5.5, 3.5))
    epochs = np.arange(1, len(history["train"]) + 1)
    ax.plot(epochs, history["train"], "o-", color=COLORS[0], linewidth=2, markersize=8, label="training images")
    ax.plot(epochs, history["val"], "o-", color=COLORS[1], linewidth=2, markersize=8, label="validation images")
    ax.set_xlabel("epoch")
    ax.set_ylabel("L1 loss")
    ax.set_title(title, loc="left")
    ax.set_xticks(epochs)
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False)
    tidy(ax)
    plt.tight_layout()
    plt.show()


def load_pretrained(model, filename):
    """Download weights trained with this notebook's settings from the module's GitHub repository."""
    url = f"https://raw.githubusercontent.com/anupbagale/mri-recon-tutorial/main/weights/{filename}"
    urllib.request.urlretrieve(url, filename)
    model.load_state_dict(torch.load(filename, map_location=device))
    return model.to(device)

In [ ]:
# Set to True to skip training and download weights trained with exactly these settings.
USE_PRETRAINED = False

> 🤔 **Predict:** How should the training and validation losses behave over the epochs? What would it mean if the training loss kept falling but the validation loss started rising?

In [ ]:
if USE_PRETRAINED:
    unet = load_pretrained(UNet(), "unet_r4.pt")
else:
    unet_history = train_model(unet, epochs=6)
    plot_history(unet_history, "U-Net: both losses fall")

Both losses fall together, so the network is learning things that also hold for images it has not seen. If the validation loss rose while the training loss fell, the network would be **overfitting**: memorizing the training images instead of learning general rules. Overfitting is most common when the training set is small compared with the size of the network.

## 5. How well does the U-Net work?

> 🤔 **Predict:** Notebook 3's compressed sensing roughly halved the zero-filled error. Will the U-Net do better or worse? And on the Shepp-Logan phantom, which it has never seen?

In [ ]:
zf_test = np.abs(zero_filled[test])
unet_test = predict(unet, zero_filled[test], measured[test])
unet_sl = predict(unet, sl_zero_filled[None], sl_measured[None])[0]

def mean_nrmse(recons, truths):
    return np.mean([nrmse(r, t) for r, t in zip(recons, truths)])

print(f"{'':<16}{'test phantoms':>15}{'Shepp-Logan':>13}   (NRMSE)")
print(f"{'zero-filled':<16}{mean_nrmse(zf_test, truth[test]):>15.3f}{nrmse(np.abs(sl_zero_filled), sl_truth):>13.3f}")
print(f"{'U-Net':<16}{mean_nrmse(unet_test, truth[test]):>15.3f}{nrmse(unet_sl, sl_truth):>13.3f}")

i = 0  # first test phantom; try other numbers
show([truth[test][i], zf_test[i], unet_test[i]], ["ground truth", "zero-filled", "U-Net"], vmin=0, vmax=1)
show([sl_truth, np.abs(sl_zero_filled), unet_sl], ["Shepp-Logan (never seen)", "zero-filled", "U-Net"], vmin=0, vmax=1)

**Interpretation:** the U-Net more than halves the error on the test phantoms (from about 0.34 to about 0.16). It also improves the Shepp-Logan phantom (from about 0.38 to about 0.22), even though that phantom looks different from anything in the training set: the network learned something general about undersampling artifacts.

But look at the images before celebrating. Much of the streaking is gone, yet the internal structures are **blurred**, faint rings remain, and small details are hard to see. Remember from Notebook 2: NRMSE rewards blur. Our U-Net trained for only a few minutes on 1000 phantoms; real U-Nets train far longer on much more data. Keep this blurriness in mind for the comparison below.

## 6. Adding data consistency: an unrolled network

The U-Net has a weakness: it never looks at the measured k-space. Its output may **disagree with the measurements**, and nothing stops it from changing parts of the image that were actually measured.

Notebook 3's ISTA had the answer: alternate "make it look like a real image" with "put the measured lines back". An **unrolled network** (e.g. Schlemper et al., 2018; Hammernik et al., 2018) does exactly that, but replaces soft-thresholding with a **small learned CNN**:

$$ x \;\rightarrow\; \text{CNN}_1 \;\rightarrow\; \text{DC} \;\rightarrow\; \text{CNN}_2 \;\rightarrow\; \text{DC} \;\rightarrow\; \cdots $$

where **DC** is the data consistency step from ISTA. Each CNN only has to fix what is still wrong, and after every step the image is guaranteed to agree with what was measured. It is "ISTA with a learned denoiser", trained end to end.

> 🤔 **Predict:** Our unrolled network has 4 small CNNs and far fewer weights than the U-Net. Will it do better or worse?

In [ ]:
def tfft2c(x):
    """Centered, orthonormal 2D FFT in PyTorch (same as fft2c)."""
    return torch.fft.fftshift(torch.fft.fft2(torch.fft.ifftshift(x, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))


def tifft2c(k):
    """Centered, orthonormal inverse 2D FFT in PyTorch (same as ifft2c)."""
    return torch.fft.fftshift(torch.fft.ifft2(torch.fft.ifftshift(k, dim=(-2, -1)), norm="ortho"), dim=(-2, -1))


def data_consistency(x, k_measured):
    """Put the measured k-space lines back, exactly as in Notebook 3 (step 1 of ISTA)."""
    image = torch.complex(x[:, 0], x[:, 1])
    k = torch.where(MASK, k_measured, tfft2c(image))
    image = tifft2c(k)
    return torch.stack([image.real, image.imag], dim=1)


class SmallCNN(nn.Module):
    """Five 3x3 convolutions that predict a correction to their input."""

    def __init__(self, channels=2, width=16, depth=5):
        super().__init__()
        layers = [nn.Conv2d(channels, width, 3, padding=1), nn.ReLU(inplace=True)]
        for _ in range(depth - 2):
            layers += [nn.Conv2d(width, width, 3, padding=1), nn.ReLU(inplace=True)]
        layers += [nn.Conv2d(width, channels, 3, padding=1)]
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return x + self.net(x)


class UnrolledNet(nn.Module):
    """A 'learned ISTA': several rounds of (small CNN -> data consistency)."""
    uses_kspace = True

    def __init__(self, cascades=4):
        super().__init__()
        self.cnns = nn.ModuleList([SmallCNN() for _ in range(cascades)])

    def forward(self, x, k_measured):
        for cnn in self.cnns:
            x = cnn(x)                                  # learned step: make it look like a real image
            x = data_consistency(x, k_measured)         # fixed step: agree with the measurements
        return x

In [ ]:
unrolled = UnrolledNet().to(device)
print(f"Unrolled network with {count_parameters(unrolled):,} trainable weights "
      f"(U-Net: {count_parameters(unet):,})")

if USE_PRETRAINED:
    unrolled = load_pretrained(UnrolledNet(), "unrolled_r4.pt")
else:
    unrolled_history = train_model(unrolled, epochs=4)
    plot_history(unrolled_history, "Unrolled network: both losses fall")

## 7. The full comparison

Now compare all four methods on the same images. Compressed sensing is slow, so we run it on the first 20 test phantoms only and compare everything on those 20.

> 🤔 **Predict:** Rank the four methods by error, and by speed.

In [ ]:
n_compare = 20
subset = slice(N_TRAIN + N_VAL, N_TRAIN + N_VAL + n_compare)
truths = truth[subset]

timings, results = {}, {}

start = time.time()
results["zero-filled"] = np.abs(ifft2c(measured[subset]))
timings["zero-filled"] = (time.time() - start) / n_compare

start = time.time()
results["compressed sensing"] = np.stack([np.abs(cs_reconstruct(m, lines)) for m in measured[subset]])
timings["compressed sensing"] = (time.time() - start) / n_compare

for name, model in [("U-Net", unet), ("unrolled network", unrolled)]:
    start = time.time()
    results[name] = predict(model, zero_filled[subset], measured[subset])
    timings[name] = (time.time() - start) / n_compare

print(f"{'method':<20}{'NRMSE':>8}{'time per image':>17}")
for name in results:
    print(f"{name:<20}{mean_nrmse(results[name], truths):>8.3f}{timings[name] * 1000:>14.1f} ms")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.2))
names = list(results)
errors = [mean_nrmse(results[n], truths) for n in names]
bars = ax.barh(names[::-1], errors[::-1], color=COLORS[0], height=0.6)
ax.bar_label(bars, fmt="%.3f", padding=4)
ax.set_xlabel("mean NRMSE on 20 test phantoms (lower is better)")
ax.set_title("The unrolled network has by far the lowest error", loc="left")
ax.set_xlim(0, max(errors) * 1.2)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
def show_with_zoom(images, titles, box):
    """Show images, then the same images zoomed into box = (row0, row1, col0, col1)."""
    r0, r1, c0, c1 = box
    show(images, titles, vmin=0, vmax=1)
    show([im[r0:r1, c0:c1] for im in images], [f"{t} (zoom)" for t in titles], vmin=0, vmax=0.6)


i = 0
row = [truths[i]] + [results[n][i] for n in names]
show_with_zoom(row, ["ground truth"] + names, box=(30, 98, 30, 98))

Look at the small **dark lesion** in the zoom, just below the bright structure in the middle. Instead of trusting our eyes, measure how much darker it is than its surroundings in each reconstruction:

In [ ]:
from scipy.ndimage import uniform_filter

def local_contrast(img, r, c, size=7):
    # how much darker pixel (r, c) is than the average of its size x size neighborhood
    return uniform_filter(img, size)[r, c] - img[r, c]

# find the darkest small spot inside the head of the ground truth
t = truths[0]
r, c = np.unravel_index(np.argmax((uniform_filter(t, 7) - t) * (t < 0.08)), t.shape)
print(f"Lesion at row {r}, column {c}.  True contrast: {local_contrast(t, r, c):.2f}")
for name in names:
    print(f"{name:<20} contrast {local_contrast(results[name][0], r, c):.2f}")

**Interpretation:**

- **Speed.** Both networks reconstruct an image in about 10–15 ms on a CPU (less on a GPU). Compressed sensing needs hundreds of iterations and is roughly 30 times slower.
- **The U-Net vs compressed sensing.** The U-Net has the lower NRMSE, but the lesion measurement shows what the number hides: the U-Net keeps only a small fraction of the lesion's contrast, while compressed sensing keeps most of it. A smooth, slightly blurred image scores well on NRMSE even when it loses a small finding. **A lower error does not guarantee a better diagnostic image.** (Part of the U-Net's weakness is our short training; Exercise 1 tests how much.)
- **The unrolled network** has by far the lowest error, about a sixth of the zero-filled error, keeps the lesion's full contrast, and shows sharp boundaries close to the ground truth. It does this with about 16 times fewer weights than the U-Net. Using the measured data at every step makes the learning problem much easier: each small CNN only has to fix what data consistency cannot.

Finally, the Shepp-Logan phantom, which neither network saw during training:

In [ ]:
sl_row = [sl_truth, np.abs(sl_zero_filled), np.abs(cs_reconstruct(sl_measured, lines)),
          predict(unet, sl_zero_filled[None], sl_measured[None])[0],
          predict(unrolled, sl_zero_filled[None], sl_measured[None])[0]]
for name, img in zip(names, sl_row[1:]):
    print(f"{name:<20} NRMSE on Shepp-Logan: {nrmse(img, sl_truth):.3f}")
show_with_zoom(sl_row, ["ground truth"] + names, box=(85, 115, 45, 85))

The ranking is the same on the Shepp-Logan phantom. In the zoom on the three small ellipses, the U-Net blurs them until they barely stand out from the background, compressed sensing breaks them into blocks, and the unrolled network keeps three separate ellipses at nearly their true brightness. **Data consistency protects details that were actually measured:** the network cannot simply paint over them, because every data consistency step puts the measured k-space lines back.

## 8. What deep learning costs, and what can go wrong

The unrolled network is both the most accurate and, once trained, among the fastest methods in this notebook. That is why deep learning reconstruction is now offered by the major MRI vendors. But the comparison hides some costs:

1. **Training data.** We needed 1000 fully sampled examples. For real scans, every one is a full-length scan of a real person, and the network is only as good as the variety in that data.
2. **Training.** Real networks train for days on large GPUs.
3. **It learns only what it saw.** Our networks were trained on one kind of phantom, one mask and R = 4. What happens when the images, the mask or the noise level change? And can a network add structures that are not really there? That is **Notebook 5**: evaluation and pitfalls.

Before moving on, the next cell saves the trained weights so Notebook 5 can reuse them. (In Colab, files disappear when the session ends; download them from the file panel on the left if you want to keep them.)

In [ ]:
torch.save(unet.state_dict(), "unet_r4.pt")
torch.save(unrolled.state_dict(), "unrolled_r4.pt")
print("Saved unet_r4.pt and unrolled_r4.pt")

## Exercises

Try each exercise before opening the hint. Solutions are in `solutions/04_solutions.ipynb`.

### Exercise 1: was the U-Net trained long enough?

In Section 4, the validation loss was still falling after 6 epochs. Train the U-Net for **6 more epochs** (`train_model(unet, epochs=6, seed=1)` continues from its current weights) and compare the test NRMSE, the images and the lesion contrast from Section 7 before and after. What does this tell you about comparing methods fairly?

<details><summary>Hint</summary>

Save the "before" results first, because training changes <code>unet</code>. For the lesion, reuse <code>local_contrast</code> with the same <code>r, c</code> as in Section 7.
</details>

In [ ]:
# Exercise 1: your code here


### Exercise 2: data consistency after the fact

Take the trained U-Net's complex output (before the magnitude) and apply one data consistency step to it, without any retraining. Does the test NRMSE improve? Compare with the unrolled network.

<details><summary>Hint</summary>

Use <code>run_model(unet, x, k)</code> to get the 2-channel output, then <code>data_consistency(output, k)</code>. Both need tensors on <code>device</code>.
</details>

In [ ]:
# Exercise 2: your code here


### Exercise 3: what did the network learn?

For one test phantom, compare the **true artifact** (zero-filled image minus ground truth) with the **correction the U-Net predicts** (U-Net output minus zero-filled input). Display both side by side. How similar are they? Where do they differ?

In [ ]:
# Exercise 3: your code here


### Exercise 4: speed in practice

A typical brain scan has about 30 slices, and a 3D scan can have more than 150. Using the times per image from Section 7, estimate how long each method would take for 30 and for 150 slices. Real images are also larger than 128 × 128: if the time grows with the number of pixels, how long would compressed sensing take for 150 slices at 256 × 256?

In [ ]:
# Exercise 4: your code here


## Summary

- **Supervised learning** trains a network on pairs of undersampled and fully sampled images, so it learns what real images look like instead of relying on a hand-made rule.
- A **U-Net** removes much of the undersampling artifact from zero-filled images in a single fast pass.
- An **unrolled network** alternates small learned CNNs with **data consistency**. It is a learned version of ISTA, uses the measurements at every step, and was the most accurate method here with far fewer weights.
- Once trained, networks reconstruct in milliseconds, far faster than iterative compressed sensing.
- The price is **training data** and a model that only knows what it was trained on. Notebook 5 tests exactly that.

**Next:** Notebook 5, *Evaluation and pitfalls*.

## References

1. Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional networks for biomedical image segmentation. *MICCAI 2015*, LNCS 9351, 234–241.
2. Schlemper, J., Caballero, J., Hajnal, J. V., Price, A. N., & Rueckert, D. (2018). A deep cascade of convolutional neural networks for dynamic MR image reconstruction. *IEEE Transactions on Medical Imaging*, 37(2), 491–503.
3. Hammernik, K., Klatzer, T., Kobler, E., Recht, M. P., Sodickson, D. K., Pock, T., & Knoll, F. (2018). Learning a variational network for reconstruction of accelerated MRI data. *Magnetic Resonance in Medicine*, 79(6), 3055–3071.
4. Zbontar, J., et al. (2018). fastMRI: An open dataset and benchmarks for accelerated MRI. *arXiv:1811.08839*.
5. Knoll, F., et al. (2020). Deep-learning methods for parallel magnetic resonance imaging reconstruction: A survey of the current approaches, trends, and issues. *IEEE Signal Processing Magazine*, 37(1), 128–140.

---
*Text and figures: CC-BY 4.0. Code: MIT License.*